In [1]:
import cv2

In [2]:
camera = cv2.VideoCapture(0)

if not camera.isOpened():
    print("Erro ao abrir webcam.")
    exit()

ret, frame_anterior = camera.read()

if not ret:
    print("Erro ao capturar frame inicial.")
    exit()


# Conversão para cinza
frame_anterior = cv2.cvtColor(
    frame_anterior,
    cv2.COLOR_BGR2GRAY
)

#Aplicar o blur
frame_anterior = cv2.GaussianBlur(
    frame_anterior,
    (21, 21),
    0
)

while True:
    ret, frame = camera.read()

    if not ret:
        print("Erro ao capturar frame inicial.")
        exit()

    # Conversão para cinza
    frame_cinza = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    #Aplicar o blur
    frame_atual_blur = cv2.GaussianBlur(
        frame_cinza,
        (21, 21),
        0
    )

    frame_diferenca = cv2.absdiff(
        frame_anterior,
        frame_atual_blur
    )

    # Threshold
    _, thresh = cv2.threshold(
          frame_diferenca,
          25, 255,
          cv2.THRESH_BINARY
    )

    # Dilatação
    frame_dilatado = cv2.dilate(
        thresh,
        None,
        2
    )

    # Contornos
    contornos, _ = cv2.findContours(
        frame_dilatado,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Análise de Contornos
    for contorno in contornos:
        area = cv2.contourArea(contorno)

        if area < 1000:
            continue

        x, y, w, h = cv2.boundingRect(contorno)

        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            "Movimento Detectado",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

    # Exibir
    cv2.imshow("Frame Original", frame)
    cv2.imshow("Diferença", frame_diferenca)
    cv2.imshow("Threshold", frame_dilatado)

    frame_anterior = frame_atual_blur.copy()

    tecla = cv2.waitKey(3) & 0xFF

    if tecla == 27:
        break

camera.release()
cv2.destroyAllWindows()